In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

# Load dataset
data = pd.read_csv("data.csv")

# Convert text to numbers
le = LabelEncoder()
for col in data.columns:
    data[col] = le.fit_transform(data[col])

# Input and Output
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

# Train Decision Tree
model = DecisionTreeClassifier()
model.fit(X, y)

# Predict using same data
pred = model.predict(X)

# Print Predictions
print("Predictions:")
print(pred)

# Draw Decision Tree
plot_tree(model, feature_names=X.columns, class_names=["No", "Yes"], filled=True)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data.csv")
x = df.iloc[:, 1:-1]   # Ignore RID column
y = df.iloc[:, -1]

def fit(x, y):
    classes = y.unique()
    priors = {}
    likelihoods = {}

    rows = len(y)
    for c in classes:
        priors[c] = len(y[y == c]) / rows

    columns = x.columns
    for features in columns:
        likelihoods[features] = {}
        for value in x[features].unique():
            likelihoods[features][value] = {}
            for c in classes:
                likelihoods[features][value][c] = len(x[(x[features] == value) & (y == c)]) / len(y[y == c])
                
    return classes, priors, likelihoods

def predict(x_test, classes, priors, likelihoods):
    predictions = []
    
    for index, row in x_test.iterrows():
        class_scores = {}
        
        for c in classes:
            score = priors[c]
            
            for features in x_test.columns:
                value = row[features]
                
                if value in likelihoods[features]:
                    score = score * likelihoods[features][value][c]
                else:
                    score = score * 0 
                    
            class_scores[c] = score
        
        best_class = max(class_scores, key=class_scores.get)
        predictions.append(best_class)
        
    return predictions

if __name__ == "__main__":
    classes, priors, likelihoods = fit(x, y)
    results = predict(x, classes, priors, likelihoods)
    print(results)

In [ ]:
import pandas as pd

# Read dataset
df = pd.read_csv("data.csv")

# Features and Class
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

classes = y.unique()

# ------------------ Prior ------------------

priors = {}

for c in classes:
    priors[c] = len(y[y == c]) / len(y)

# ------------------ Test Tuple ------------------

test = {
    "Age": "Youth",
    "Income": "Low",
    "Student": "Yes",
    "CR": "Fair"
}

# ------------------ Prediction ------------------

posterior = {}

for c in classes:

    prob = priors[c]

    for feature in X.columns:

        value = test[feature]

        total = len(df[y == c])

        count = len(df[(df[feature] == value) & (y == c)])

        # Laplace Smoothing
        prob *= (count + 1) / (total + len(X[feature].unique()))

    posterior[c] = prob

print("Posterior Probabilities:")
print(posterior)

prediction = max(posterior, key=posterior.get)

print("\nPredicted Class =", prediction)

In [ ]:
import pandas as pd

# Read dataset
df = pd.read_csv("data.csv")

X = df.iloc[:, 1:-1]      # Ignore RID
y = df.iloc[:, -1]        # Class Label

classes = y.unique()

# Predict all rows
for i in range(len(X)):

    scores = {}

    for c in classes:

        # Prior Probability
        prob = len(y[y == c]) / len(y)

        # Likelihood
        for feature in X.columns:

            value = X.iloc[i][feature]

            count = len(df[(df[feature] == value) & (y == c)])
            total = len(y[y == c])

            # Laplace Smoothing
            prob *= (count + 1) / (total + len(X[feature].unique()))

        scores[c] = prob

    prediction = max(scores, key=scores.get)

    print("Tuple", i + 1, ":", prediction)